# 08주차 실습: Spark ML 학습·튜닝·저장

강의: [PDF](../pdf/week08_spark_ml_pipeline.pdf) · **PDF 실제 페이지 6–25** (인쇄 번호와 다를 수 있음)

## 학습 목표
전처리를 포함한 Pipeline을 검증 세트에서 선택하고 저장·재로딩한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 데이터 분할과 Pipeline

In [ ]:
from labkit.common import spark_session
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
spark=spark_session('week08')
df=spark.read.csv(str(DATA/'ctr.csv'),header=True,inferSchema=True).withColumnRenamed('clicked','label')
train,valid,test=df.randomSplit([.6,.2,.2],seed=42)
indexer=StringIndexer(inputCol='device',outputCol='device_idx',handleInvalid='keep')
encoder=OneHotEncoder(inputCol='device_idx',outputCol='device_vec',handleInvalid='keep')
assembler=VectorAssembler(inputCols=['age','hour','bid_price','ad_quality','popularity','device_vec'],outputCol='raw_features')
scaler=StandardScaler(inputCol='raw_features',outputCol='features')
evaluator=BinaryClassificationEvaluator(metricName='areaUnderROC')
trials=[]; models=[]
for reg in [.01,.1]:
    pipeline=Pipeline(stages=[indexer,encoder,assembler,scaler,LogisticRegression(regParam=reg,maxIter=30)])
    model=pipeline.fit(train); auc=evaluator.evaluate(model.transform(valid))
    trials.append({'regParam':reg,'validation_auc':auc}); models.append(model)
display(pd.DataFrame(trials))

## 홀드아웃 평가와 재로딩

In [ ]:
import uuid
best=models[int(np.argmax([r['validation_auc'] for r in trials]))]
out=output_dir('week08')/uuid.uuid4().hex
best.write().save(str(out/'model'))
loaded=PipelineModel.load(str(out/'model'))
auc=evaluator.evaluate(loaded.transform(test))
assert abs(auc-evaluator.evaluate(best.transform(test))) < 1e-10
save_json(out/'metrics.json',{'trials':trials,'test_auc':auc,'spark_version':spark.version})
print('test AUC:',auc)
spark.stop()

## 확장 과제 및 제출
ParamGridBuilder와 CrossValidator로 학습 세트 내부 CV를 추가하세요. 저장된 PipelineModel.transform을 7주차 readStream의 광고 스키마에 적용하는 흐름을 설계하세요. 스트림 예측과 온라인 학습을 구분하세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.